# PDF Editing and Plan Processing Utilities

Tools for working with ODOT plan sets: bookmark extraction,
section splitting, sheet type identification (regex and fuzzy matching),
interactive label editing, TOC generation, and Bluebeam integration.


# Plan Scanning With Python

## Getting the Bookmarks (Outlines) of a Document

In [11]:
path_to_pdf = r"X:\Bridge Design Resources\Projects\District 1\WYA\WYA-23-3.47\DRP - Comments\Backups\2025-07-08.121424_WYA-23-3.47 Stage 2 Plans - DRP Comments.pdf"
doc2 = fitz.open(path_to_pdf) # open a document
toc = doc2.get_toc(False)

In [12]:
for index, bookmark in enumerate(toc):
    if bookmark[1][:7] == 'Chapter':
        print(f"Index: {index}, Title: {bookmark[1]}")

In [13]:
toc[0]

In [14]:
for bookmark in toc:
    print(bookmark[1])

# Splitting Plans by Section

In [13]:
import pdfplumber

In [14]:
# Title Page Definitons
def check_page(split_text, page_number):
    # Patterns to match the "Title Page" (Title text appears backwards)
    if 'ECNANETNIAM' in split_text and 'CIFFART' in split_text:
        print(f"MOT Found: {page_number}")
        return {"MOT": page_number}
    elif "ELTIT" in split_text and "TEEHS" in split_text:
        print(f"Title page found: {page_number}")
        return {"Title Page": page_number}
    elif 'NALP' in split_text and 'CITAMEHCS' in split_text:
        print(f"Schematic Plan Found: {page_number}")
        return {"Schematic Plan": page_number}
    elif 'SNOITCES' in split_text and 'LACIPYT' in split_text:
        print(f"Typical Sections Found: {page_number}")
        return {"Typical Sections Plan": page_number}
    elif 'LARENEG' in split_text and 'SETON' in split_text:
        print(f"General Notes Found: {page_number}")
        return {"General Notes": page_number}
    else:
        pass

In [15]:
def extract_sections_from_pdf(pdf_path, output_folder):
    """
    Extracts sections from a PDF based on keywords and saves them as separate text files.
    
    Args:
        pdf_path (str): Path to the input PDF file.
        output_folder (str): Folder to save the extracted sections.
        section_keywords (list): List of keywords that identify the start of a section.
    """
    sections = {}
    current_section = None
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text().split('\n')
            
            if not text:
                continue  # Skip empty pages

            page_type = check_page(text, page_number)

            if page_type:
                return_key = next(iter(page_type))
                return_page = page_type[return_key]
            
                if return_key not in sections:
                    sections[return_key] = [return_page]
                else:
                    sections[return_key].append(return_page)
    return sections

In [16]:
# Example usage
pdf_path = r"X:\Bridge Design Resources\Projects\District 1\WYA\WYA-23-3.47\Stage 2 - Submission\2025-07-08.121424_WYA-23-3.47 Stage 2 Plans.pdf"
output_folder = r"X:\Bridge Design Resources\Projects\District 1\WYA\WYA-23-3.47\Python\output"

extract_sections_from_pdf(pdf_path, output_folder)

## Fuzzy Match

In [17]:
import fitz  # PyMuPDF
from rapidfuzz import fuzz

In [18]:
def extract_text_pymupdf(pdf_path):
    """
    Extracts text from a PDF file using PyMuPDF.

    Args:
        pdf_path (str): Path to the PDF file.

    Returns:
        dict: A dictionary where keys are page numbers and values are text content from each page.
    """
    # Open the PDF file
    doc = fitz.open(pdf_path)

    # Dictionary to store page text
    text_by_page = {}

    # Iterate through pages and extract text
    for page_number in range(len(doc)):
        page = doc.load_page(page_number)  # Load each page
        text = page.get_text()  # Extract text from the page
        text_by_page[page_number + 1] = text  # Store text with a 1-based page index

    # Close the document
    doc.close()

    return text_by_page

In [19]:
def fuzzy_partial_match(query, text_list, threshold=85, min_length=3):
    """
    Check if the query approximately matches any string in the list,
    allowing for partial matches with additional constraints.

    Args:
        query (str): The substring to search for.
        text_list (list): The list of text strings to compare against.
        threshold (int): Minimum similarity score for matching (default is 80).
        min_length (int): Minimum length of a string to be considered in comparisons.

    Returns:
        bool: True if a partial match is found, otherwise False.
    """
    for text in text_list:
        # Skip very short strings to avoid false positives (e.g., 'P')
        if len(text) < min_length:
            continue

        # Calculate the partial match ratio
        partial_ratio = fuzz.partial_ratio(query, text)

        # Only return True for matches exceeding the threshold
        if partial_ratio >= threshold:
            return True

    return False

In [20]:
text_dict = extract_text_pymupdf(pdf_path)

# Iterate the pages and try to accurately identify the page type
for page_num, text in text_dict.items():
    # Title Sheet 
    if fuzzy_partial_match("TITLE ", text.split('\n')):
        print(f"Title Sheet: {page_num}")

    # MOT Pages 
    elif fuzzy_partial_match("MAINTENANCE OF TRAFFIC", text.split('\n')):
        print(f"MOT Pages : {page_num}")

    # Schematic Plan
    elif fuzzy_partial_match("SCHEMATIC PLAN", text.split('\n')):
        print(f"Schematic Plan: {page_num}")

    # Typical Sections
    elif fuzzy_partial_match("TYPICAL SECTIONS", text.split('\n')):
        print(f"Typical Sections: {page_num}")

    # General Notes
    elif fuzzy_partial_match("GENERAL NOTES", text.split('\n')):
        print(f"General Notes: {page_num}")

    # Plan and Profile
    elif fuzzy_partial_match("PLAN AND PROFILE", text.split('\n')):
        print(f"Plan and Profile: {page_num}")

In [21]:
odot_sheets = {
    # Geotechnical
    "YL": "Geotechnical: Geohazard Boring Logs",
    "YC": "Geotechnical: Geohazard Cover",
    "YX": "Geotechnical: Geohazard Cross Sections",
    "YD": "Geotechnical: Geohazard Lab Data",
    "YP": "Geotechnical: Geohazard Plan and Profile",
    "YF": "Geotechnical: Geohazard Profile",
    "IC": "Geotechnical: Geotechnical Profile Cover",
    "IX": "Geotechnical: Geotechnical Profile Cross Sections",
    "ID": "Geotechnical: Geotechnical Profile Lab Data",
    "IP": "Geotechnical: Geotechnical Profile, Plan and Profile or Plan",
    "IF": "Geotechnical: Geotechnical Profile, Profile Only",
    "ZL": "Geotechnical: Structure Foundation Exploration Boring Logs",
    "ZC": "Geotechnical: Structure Foundation Exploration Cover",
    "ZD": "Geotechnical: Structure Foundation Exploration Lab Data",
    "ZP": "Geotechnical: Structure Foundation Exploration Plan and Profile",
    "ZF": "Geotechnical: Structure Foundation Exploration Profile",

    # Drainage
    "DC": "Drainage: Culvert Details",
    "DD": "Drainage: Details",
    "DE": "Drainage: Erosion Control",
    "DM": "Drainage: Miscellaneous Details",
    "DN": "Drainage: Notes",
    "DP": "Drainage: Plan and Profile or Plan",
    "DF": "Drainage: Profile",
    "DQ": "Drainage: Quantity Table",
    "DB": "Drainage: Schematic Plan",
    "DS": "Drainage: Sub-Summary",

    # Landscaping
    "PD": "Landscaping: Details",
    "PM": "Landscaping: Miscellaneous Details",
    "PN": "Landscaping: Notes",
    "PP": "Landscaping: Plan",
    "PB": "Landscaping: Schematic Plan",
    "PS": "Landscaping: Sub-Summary",
    
    # Lighting
    "LC": "Lighting: Circuit Diagrams",
    "LD": "Lighting: Details",
    "LE": "Lighting: Elevation Views",
    "LG": "Lighting: General Summary",
    "LM": "Lighting: Miscellaneous",
    "LN": "Lighting: Notes",
    "LP": "Lighting: Plan",
    "LQ": "Lighting: Quantity Table",
    "LB": "Lighting: Schematic Plan",
    "LS": "Lighting: Sub-Summary",
    
    # Maintainence of Traffic
    "XM": "MOT: Cross Sections",
    "MD": "MOT: Detour Plan",
    "MM": "MOT: Miscellaneous",
    "MN": "MOT: Notes",
    "MP": "MOT: Phase Plan and Profile or Plan",
    "MH": "MOT: Phase Details",
    "MF": "MOT: Profile",
    "MQ": "MOT: Quantity Table",
    "MB": "MOT: Schematic Plan",
    "MS": "MOT: Sub-Summary",
    "MY": "MOT: Typical Sections",
    
    # Right of Way
    "RC": "RoW: Centerline Plat",
    "RL": "RoW: Legend",
    "RM": "RoW: Property Map",
    "RR": "RoW: Railroad Plat",
    "RB": "RoW: RW Boundary",
    "RD": "RoW: RW Detail",
    "RT": "RoW: RW Topo",
    "RS": "RoW: Summary of Additional RW",
    
    # Roadway
    "GC": "Roadway: Calculations/Computations",
    "XS": "Roadway: Cross Sections",
    "GD": "Roadway: Drive Details",
    "GX": "Roadway: Fencing Plan",
    "GN": "Roadway: General Notes",
    "GG": "Roadway: General Summary",
    "XG": "Roadway: Grading Plan",
    "GR": "Roadway: Guardrail/Barrier Details",
    "GI": "Roadway: Intersection/Interchange Details",
    "GJ": "Roadway: Maintenance Data",
    "GM": "Roadway: Miscellaneous",
    "GA": "Roadway: Pavement Details",
    "GP": "Roadway: Plan and Profile or Plan",
    "GF": "Roadway: Profile",
    "GQ": "Roadway: Quantity Table",
    "GB": "Roadway: Schematic Plan",
    "GS": "Roadway: Sub-Summary",
    "GE": "Roadway: Superelevation Table",
    "GT": "Roadway: Title Sheet",
    "GU": "Roadway: Signature Sheet",
    "GY": "Roadway: Typical Sections",

    # Signals
    "CD": "Signals: Details",
    "CG": "Signals: General Summary",
    "CM": "Signals: Miscellaneous",
    "CN": "Signals: Notes",
    "CP": "Signals: Plan",
    "CQ": "Signals: Quantity Table",
    "CS": "Signals: Sub-Summary",
    
    # Traffic Control Sheets
    "TC": "Traffic Control: Calculations/Computations",
    "TD": "Traffic Control: Details",
    "TE": "Traffic Control: Elevation Views",
    "TN": "Traffic Control: General Notes",
    "TG": "Traffic Control: General Summary",
    "TM": "Traffic Control: Miscellaneous",
    "TP": "Traffic Control: Plan",
    "TQ": "Traffic Control: Quantity Table",
    "TB": "Traffic Control: Schematic Plan",
    "TS": "Traffic Control: Sub-Summary",

    # Utilities
    "UC": "Utilities: Calculations/Computations",
    "UD": "Utilities: Details",
    "UE": "Utilities: Elevation Views",
    "UG": "Utilities: General Summary",
    "UM": "Utilities: Miscellaneous",
    "UN": "Utilities: Notes",
    "UP": "Utilities: Plan and Profile or Plan",
    "UF": "Utilities: Profile",
    "UQ": "Utilities: Quantity Table",
    "UB": "Utilities: Schematic Plan",
    "US": "Utilities: Sub-Summary",
    
    # Wall Sheets
    "WC": "Wall: Calculations/Computations",
    "WX": "Wall: Cross Sections",
    "WD": "Wall: Details",
    "WE": "Wall: Elevation",
    "WQ": "Wall: Estimated Quantities",
    "WT": "Wall: Foundation",
    "WM": "Wall: Miscellaneous",
    "WN": "Wall: Notes",
    "WP": "Wall: Plan and Profile or Plan",
    "WF": "Wall: Profile",
    "WB": "Wall: Schematic Plan",
    "WH": "Wall: Sheeting",
    "WL": "Wall: Steel List",
    "WS": "Wall: Sub-Summary",
    "WY": "Wall: Typical Section",

    # Structures
    "SB": "Structures: Bearing",
    "SD": "Structures: Deck Plan",
    "SQ": "Structures: Estimated Quantities",
    "SX": "Structures: Expansion Device Details",
    "SF": "Structures: Forward Abutment",
    "SO": "Structures: Foundation Plan",
    "SN": "Structures: General Notes",
    "SG": "Structures: General Plan",
    "SM": "Structures: Miscellaneous Details",
    "SI": "Structures: Piers",
    "SA": "Structures: Railing",
    "SR": "Structures: Rear Abutment",
    "SL": "Structures: Reinforcing Steel List",
    "SV": "Structures: Removal",
    "SH": "Structures: Sheeting",
    "SP": "Structures: Site Plan",
    "SC": "Structures: Staged Construction Details",
    "SS": "Structures: Superstructure Detail",
    "ST": "Structures: Transverse Section",
}

## Regex

In [22]:
import re

In [37]:
patterns = {
    # Geotechnical
    "Geotechnical: Geohazard Boring Logs": r"\d{6}_YL\d{3}",
    "Geotechnical: Geohazard Cover": r"\d{6}_YC\d{3}",
    "Geotechnical: Geohazard Cross Sections": r"\d{6}_YX\d{3}",
    "Geotechnical: Geohazard Lab Data": r"\d{6}_YD\d{3}",
    "Geotechnical: Geohazard Plan and Profile": r"\d{6}_YP\d{3}",
    "Geotechnical: Geohazard Profile": r"\d{6}_YF\d{3}",
    "Geotechnical: Geotechnical Profile Cover": r"\d{6}_IC\d{3}",
    "Geotechnical: Geotechnical Profile Cross Sections": r"\d{6}_IX\d{3}",
    "Geotechnical: Geotechnical Profile Lab Data": r"\d{6}_ID\d{3}",
    "Geotechnical: Geotechnical Profile, Plan and Profile or Plan": r"\d{6}_IP\d{3}",
    "Geotechnical: Geotechnical Profile, Profile Only": r"\d{6}_IF\d{3}",
    "Geotechnical: Structure Foundation Exploration Boring Logs": r"\d{6}_ZL\d{3}",
    "Geotechnical: Structure Foundation Exploration Cover": r"\d{6}_ZC\d{3}",
    "Geotechnical: Structure Foundation Exploration Lab Data": r"\d{6}_ZD\d{3}",
    "Geotechnical: Structure Foundation Exploration Plan and Profile": r"\d{6}_ZP\d{3}",
    "Geotechnical: Structure Foundation Exploration Profile": r"\d{6}_ZF\d{3}",

    # Drainage
    "Drainage: Culvert Details": r"\d{6}_DC\d{3}",
    "Drainage: Details": r"\d{6}_DD\d{3}",
    "Drainage: Erosion Control": r"\d{6}_DE\d{3}",
    "Drainage: Miscellaneous Details": r"\d{6}_DM\d{3}",
    "Drainage: Notes": r"\d{6}_DN\d{3}",
    "Drainage: Plan and Profile or Plan": r"\d{6}_DP\d{3}",
    "Drainage: Profile": r"\d{6}_DF\d{3}",
    "Drainage: Quantity Table": r"\d{6}_DQ\d{3}",
    "Drainage: Schematic Plan": r"\d{6}_DB\d{3}",
    "Drainage: Sub-Summary": r"\d{6}_DS\d{3}",

    # Landscaping
    "Landscaping: Details": r"\d{6}_PD\d{3}",
    "Landscaping: Miscellaneous Details": r"\d{6}_PM\d{3}",
    "Landscaping: Notes": r"\d{6}_PN\d{3}",
    "Landscaping: Plan": r"\d{6}_PP\d{3}",
    "Landscaping: Schematic Plan": r"\d{6}_PB\d{3}",
    "Landscaping: Sub-Summary": r"\d{6}_PS\d{3}",
    
    # Lighting
    "Lighting: Circuit Diagrams": r"\d{6}_LC\d{3}",
    "Lighting: Details": r"\d{6}_LD\d{3}",
    "Lighting: Elevation Views": r"\d{6}_LE\d{3}",
    "Lighting: General Summary": r"\d{6}_LG\d{3}",
    "Lighting: Miscellaneous": r"\d{6}_LM\d{3}",
    "Lighting: Notes": r"\d{6}_LN\d{3}",
    "Lighting: Plan": r"\d{6}_LP\d{3}",
    "Lighting: Quantity Table": r"\d{6}_LQ\d{3}",
    "Lighting: Schematic Plan": r"\d{6}_LB\d{3}",
    "Lighting: Sub-Summary": r"\d{6}_LS\d{3}",
    
    # Maintainence of Traffic
    "MOT: Cross Sections": r"\d{6}_XM\d{3}",
    "MOT: Detour Plan": r"\d{6}_MD\d{3}",
    "MOT: Miscellaneous": r"\d{6}_MM\d{3}",
    "MOT: Notes": r"\d{6}_MN\d{3}",
    "MOT: Phase Plan and Profile or Plan": r"\d{6}_MP\d{3}",
    "MOT: Phase Details": r"\d{6}_MH\d{3}",
    "MOT: Profile": r"\d{6}_MF\d{3}",
    "MOT: Quantity Table": r"\d{6}_MQ\d{3}",
    "MOT: Schematic Plan": r"\d{6}_MB\d{3}",
    "MOT: Sub-Summary": r"\d{6}_MS\d{3}",
    "MOT: Typical Sections": r"\d{6}_MY\d{3}",
    
    # Right of Way
    "RoW: Centerline Plat": r"\d{6}_RC\d{3}",
    "RoW: Legend": r"\d{6}_RL\d{3}",
    "RoW: Property Map": r"\d{6}_RM\d{3}",
    "RoW: Railroad Plat": r"\d{6}_RR\d{3}",
    "RoW: RW Boundary": r"\d{6}_RB\d{3}",
    "RoW: RW Detail": r"\d{6}_RD\d{3}",
    "RoW: RW Topo": r"\d{6}_RT\d{3}",
    "RoW: Summary of Additional RW": r"\d{6}_RS\d{3}",
    
    # Roadway
    "Roadway: Calculations/Computations": r"\d{6}_GC\d{3}",
    "Roadway: Cross Sections": r"\d{6}_XS\d{3}",
    "Roadway: Drive Details": r"\d{6}_GD\d{3}",
    "Roadway: Fencing Plan": r"\d{6}_GX\d{3}",
    "Roadway: General Notes": r"\d{6}_GN\d{3}",
    "Roadway: General Summary": r"\d{6}_GG\d{3}",
    "Roadway: Grading Plan": r"\d{6}_XG\d{3}",
    "Roadway: Guardrail/Barrier Details": r"\d{6}_GR\d{3}",
    "Roadway: Intersection/Interchange Details": r"\d{6}_GI\d{3}",
    "Roadway: Maintenance Data": r"\d{6}_GJ\d{3}",
    "Roadway: Miscellaneous": r"\d{6}_GM\d{3}",
    "Roadway: Pavement Details": r"\d{6}_GA\d{3}",
    "Roadway: Plan and Profile or Plan": r"\d{6}_GP\d{3}",
    "Roadway: Profile": r"\d{6}_GF\d{3}",
    "Roadway: Quantity Table": r"\d{6}_GQ\d{3}",
    "Roadway: Schematic Plan": r"\d{6}_GB\d{3}",
    "Roadway: Sub-Summary": r"\d{6}_GS\d{3}",
    "Roadway: Superelevation Table": r"\d{6}_GE\d{3}",
    "Roadway: Title Sheet": r"\d{6}_GT\d{3}",
    "Roadway: Signature Sheet": r"\d{6}_GU\d{3}",
    "Roadway: Typical Sections": r"\d{6}_GY\d{3}",

    # Signals
    "Signals: Details": r"\d{6}_CD\d{3}",
    "Signals: General Summary": r"\d{6}_CG\d{3}",
    "Signals: Miscellaneous": r"\d{6}_CM\d{3}",
    "Signals: Notes": r"\d{6}_CN\d{3}",
    "Signals: Plan": r"\d{6}_CP\d{3}",
    "Signals: Quantity Table": r"\d{6}_CQ\d{3}",
    "Signals: Sub-Summary": r"\d{6}_CS\d{3}",
    
    # Traffic Control Sheets
    "Traffic Control: Calculations/Computations": r"\d{6}_TC\d{3}",
    "Traffic Control: Details": r"\d{6}_TD\d{3}",
    "Traffic Control: Elevation Views": r"\d{6}_TE\d{3}",
    "Traffic Control: General Notes": r"\d{6}_TN\d{3}",
    "Traffic Control: General Summary": r"\d{6}_TG\d{3}",
    "Traffic Control: Miscellaneous": r"\d{6}_TM\d{3}",
    "Traffic Control: Plan": r"\d{6}_TP\d{3}",
    "Traffic Control: Quantity Table": r"\d{6}_TQ\d{3}",
    "Traffic Control: Schematic Plan": r"\d{6}_TB\d{3}",
    "Traffic Control: Sub-Summary": r"\d{6}_TS\d{3}",

    # Utilities
    "Utilities: Calculations/Computations": r"\d{6}_UC\d{3}",
    "Utilities: Details": r"\d{6}_UD\d{3}",
    "Utilities: Elevation Views": r"\d{6}_UE\d{3}",
    "Utilities: General Summary": r"\d{6}_UG\d{3}",
    "Utilities: Miscellaneous": r"\d{6}_UM\d{3}",
    "Utilities: Notes": r"\d{6}_UN\d{3}",
    "Utilities: Plan and Profile or Plan": r"\d{6}_UP\d{3}",
    "Utilities: Profile": r"\d{6}_UF\d{3}",
    "Utilities: Quantity Table": r"\d{6}_UQ\d{3}",
    "Utilities: Schematic Plan": r"\d{6}_UB\d{3}",
    "Utilities: Sub-Summary": r"\d{6}_US\d{3}",
    
    # Wall Sheets
    "Wall: Calculations/Computations": r"\d{6}_WC\d{3}",
    "Wall: Cross Sections": r"\d{6}_WX\d{3}",
    "Wall: Details": r"\d{6}_WD\d{3}",
    "Wall: Elevation": r"\d{6}_WE\d{3}",
    "Wall: Estimated Quantities": r"\d{6}_WQ\d{3}",
    "Wall: Foundation": r"\d{6}_WT\d{3}",
    "Wall: Miscellaneous": r"\d{6}_WM\d{3}",
    "Wall: Notes": r"\d{6}_WN\d{3}",
    "Wall: Plan and Profile or Plan": r"\d{6}_WP\d{3}",
    "Wall: Profile": r"\d{6}_WF\d{3}",
    "Wall: Schematic Plan": r"\d{6}_WB\d{3}",
    "Wall: Sheeting": r"\d{6}_WH\d{3}",
    "Wall: Steel List": r"\d{6}_WL\d{3}",
    "Wall: Sub-Summary": r"\d{6}_WS\d{3}",
    "Wall: Typical Section": r"\d{6}_WY\d{3}",

    # Structures
    "Structures: Bearing": r"\d{6}_SB\d{3}",
    "Structures: Deck Plan": r"\d{6}_SD\d{3}",
    "Structures: Estimated Quantities": r"\d{6}_SQ\d{3}",
    "Structures: Expansion Device Details": r"\d{6}_SX\d{3}",
    "Structures: Forward Abutment": r"\d{6}_SF\d{3}",
    "Structures: Foundation Plan": r"\d{6}_SO\d{3}",
    "Structures: General Notes": r"\d{6}_SN\d{3}",
    "Structures: General Plan": r"\d{6}_SG\d{3}",
    "Structures: Miscellaneous Details": r"\d{6}_SM\d{3}",
    "Structures: Piers": r"\d{6}_SI\d{3}",
    "Structures: Railing": r"\d{6}_SA\d{3}",
    "Structures: Rear Abutment": r"\d{6}_SR\d{3}",
    "Structures: Reinforcing Steel List": r"\d{6}_SL\d{3}",
    "Structures: Removal": r"\d{6}_SV\d{3}",
    "Structures: Sheeting": r"\d{6}_SH\d{3}",
    "Structures: Site Plan": r"\d{6}_SP\d{3}",
    "Structures: Staged Construction Details": r"\d{6}_SC\d{3}",
    "Structures: Superstructure Detail": r"\d{6}_SS\d{3}",
    "Structures: Transverse Section": r"\d{6}_ST\d{3}",
}

In [38]:
regex_pattern = r"\d{6}_([A-Z]{2})\d{3}"

In [39]:
test_string = "MODEL: 121424_GP103 PAPERSIZE: 17x11 DATE: 7/8/2025"

In [40]:
matches = re.findall(regex_pattern, test_string)

In [41]:
labels = {}

for page_num, text in text_dict.items():    
    matches = re.findall(regex_pattern, text)

    if matches:
        if len(matches) != 1:
            print(f"Page: {page_num}\n{matches}")
        else:
            labels[page_num] = odot_sheets[matches[0]]
    else:
        labels[page_num] = None

In [42]:
fuzzy_partial_match("PLAN AND PROFILE", text_dict[54].split('\n'))

In [43]:
labels

In [44]:
import ipywidgets as widgets
from IPython.display import display
import ast

In [45]:
# Initial dictionary - labels
results = labels

# Prepare the initial text for editing
initial_text = "\n".join(f"{key}: {value if value is not None else ''}" for key, value in results.items())

# Create a text widget
text_area = widgets.Textarea(
    value=initial_text,
    placeholder="Edit the text here...",
    description="Editor:",
    layout=widgets.Layout(width='900px', height='400px')
)

# Display the text area
display(text_area)

# Button to capture edited text
button = widgets.Button(description="Submit Changes")
output = widgets.Output()

def on_button_click(b):
    with output:
        # Read and display edited text
        edited_text = text_area.value

button.on_click(on_button_click)

# Display button and output
display(button, output)

Raw editor input in case the notebook ever gets rerun.

Since it's easier to update sheet names, we let the user do that and then  
reconvert the 2 digit codes back into their descriptions.

In [56]:
# Splits the user input into a list of key/value pairs
updated_list = text_area.value.split('\n')[0:]

# Converts the key value pairs back into a python dictionary
result_dict = {int(x.split(': ', 1)[0]): x.split(': ', 1)[1] for x in updated_list}

In [57]:
for page_num, label in result_dict.items():
    if len(label) == 2:
        labels[page_num] = odot_sheets[label]

In [58]:
labels

# Open the Sheet Descriptions to Edit in an IDE

In [49]:
import subprocess
from pprint import pformat

In [50]:
# Temporary file to edit
tmp_file = "C:\\Users\\dparks1\\Desktop\\test.txt"

# Write the file content
with open(tmp_file, 'w') as f:
    f.write(pformat(labels))

In [51]:
# Open the file in VSCode using subprocess
subprocess.run([
    "C:\\Users\\dparks1\\AppData\\Local\\JetBrains\\PyCharm 2025.1.1.1\\bin\\pycharm64.exe",  # VSCode works too
    "C:\\Users\\dparks1\\Desktop\\test.txt"
]);

# Generating Table of Contents

In [52]:
import subprocess
import os

In [53]:
def open_pdf_on_page(pdf_path, page):
    """
    Open a specific page of a PDF using subprocess. It will open with the default PDF application.

    Args:
        pdf_path (str): The absolute path to the PDF file.
        page (int): The page number to open.

    Returns:
        None
    """
    if not os.path.isfile(pdf_path):
        raise FileNotFoundError(f"The provided PDF file does not exist: {pdf_path}")

    # Bluebeam and PDF viewers (like Adobe Acrobat) support page navigation through command-line arguments.
    # This works for Adobe Acrobat or most default PDF readers. Adjust this if Bluebeam uses additional flags.

    # Command for Adobe Acrobat/Bluebeam to open a specific page
    args = [pdf_path, f'/A', f'page={page}']

    # Call subprocess to open the PDF
    subprocess.Popen(['explorer', pdf_path])

In [54]:
def generate_table_of_contents(page_dict, pdf_path):
    """
    Create a TOC from the dictionary of pages and allow launching specific pages using subprocess.

    Args:
        page_dict (dict): A dictionary where the keys are page numbers and the values are sheet names.
        pdf_path (str): The absolute path to the PDF file.

    Returns:
        None: Prints the TOC and opens pages upon interaction.
    """
    toc = []
    current_sheet = None
    start_page = None
    end_page = None
    
    # Ensure the file exists
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"The PDF file does not exist: {pdf_path}")

    # Iterate through the sorted dictionary
    for page, sheet_name in sorted(page_dict.items()):
        if sheet_name != current_sheet:
            if current_sheet is not None:
                if start_page == end_page:
                    toc.append((start_page, f"Page {start_page}: {current_sheet}"))
                else:
                    toc.append((start_page, f"Pages {start_page}-{end_page}: {current_sheet}"))
            # Start new page range
            current_sheet = sheet_name
            start_page = page
            end_page = page
        else:
            # Extend the range for the same sheet
            end_page = page

    # Add the final entry
    if current_sheet is not None:
        if start_page == end_page:
            toc.append((start_page, f"Page {start_page}: {current_sheet}"))
        else:
            toc.append((start_page, f"Pages {start_page}-{end_page}: {current_sheet}"))
    
    # Print the TOC and allow interaction to open pages
    for page, description in toc:
        print(description)

    return toc

In [55]:
pdf_path = r"X:\Bridge Design Resources\Projects\District 1\WYA\WYA-23-3.47\DRP - Comments\2025-07-08.121424_WYA-23-3.47 Stage 2 Plans - DRP Comments.pdf"

# Generate TOC and allow interaction
toc = generate_table_of_contents(labels, pdf_path)

In [90]:
from pypdf import PdfReader, PdfWriter
import os
import re

In [91]:
input_pdf_path = r"X:\Bridge Design Resources\Projects\District 1\WYA\WYA-23-3.47\DRP - Comments\Backups\WYA-23-3.47 code testing.pdf"
# This will be the path for the NEW PDF file with bookmarks added
output_pdf_path = r"X:\Bridge Design Resources\Projects\District 1\WYA\WYA-23-3.47\DRP - Comments\Backups\WYA-23-3.47 code testing_with_bookmarks.pdf"

In [92]:
def add_bookmarks_to_pdf(input_pdf_path, output_pdf_path, toc_data):
    """
    Reads an input PDF, adds bookmarks (outlines) based on a table of contents,
    and saves the result to a new output PDF. Handles single-item subgroups
    by promoting them to top-level bookmarks.

    Args:
        input_pdf_path (str): The file path of the original PDF.
        output_pdf_path (str): The file path where the new PDF with bookmarks will be saved.
        toc_data (list): A list of tuples, where each tuple is (page_number, 'Title String').
                         Page numbers are assumed to be 1-indexed.
    """
    try:
        reader = PdfReader(input_pdf_path)
        writer = PdfWriter()

        for page in reader.pages:
            writer.add_page(page)

        grouped_toc = {}
        for page_num, title_full in toc_data:
            # Regex to find text between the first and second colons (e.g., "Roadway" in "Page X: Roadway: Title")
            # This is for the main section name (e.g., 'Roadway', 'MOT', 'Structures')
            section_name_match = re.search(r':\s*([^:]+?)\s*:', title_full)
            if section_name_match:
                section_name = section_name_match.group(1).strip()
            else:
                # If there's no second colon, it might be a top-level category or a simpler format.
                # Try to extract the part after the first colon as the section name,
                # or use "Miscellaneous" if no clear structure.
                first_colon_match = re.search(r'^\s*[^:]*?:\s*([^:]+)', title_full)
                if first_colon_match:
                    section_name = first_colon_match.group(1).strip()
                else:
                    section_name = "Miscellaneous" # Fallback

            if section_name not in grouped_toc:
                grouped_toc[section_name] = []
            grouped_toc[section_name].append((page_num, title_full))

        # Add bookmarks to the PDF based on the grouped TOC
        for section_name in grouped_toc:
            section_items = grouped_toc[section_name]

            if len(section_items) == 1:
                # If there's only one item in this section, add it directly as a top-level bookmark
                page_num, title_full = section_items[0]

                # Extract the bookmark title more carefully for single items
                # If it's something like 'Page 176: Roadway: Drive Details', we want 'Roadway: Drive Details'
                # If it's 'Page 177: Drainage: Profile', we might want 'Drainage: Profile'
                # Find the first colon and take everything after it
                first_colon_pos = title_full.find(':')
                if first_colon_pos != -1:
                    bookmark_title = title_full[first_colon_pos + 1:].strip()
                else:
                    bookmark_title = title_full # Fallback if no colon

                # Trim leading 'Roadway:', 'Drainage:', etc. if it's redundant.
                # Example: "Roadway: Drive Details" -> "Drive Details" IF "Roadway" is the section_name
                if bookmark_title.lower().startswith(section_name.lower()):
                    # Try to remove the section name from the title if it starts with it
                    # e.g., "Roadway: Title Sheet" -> "Title Sheet"
                    # But be careful not to remove too much.
                    temp_title_parts = bookmark_title.split(':', 1)
                    if len(temp_title_parts) > 1 and temp_title_parts[0].strip().lower() == section_name.lower():
                        final_bookmark_title = temp_title_parts[1].strip()
                    else:
                        final_bookmark_title = bookmark_title # Keep original if complex
                else:
                    final_bookmark_title = bookmark_title

                # For single items, promote the sub-item title to be the main bookmark title.
                # We can choose to keep the section name as a prefix, or remove it if redundant.
                # Let's use the full descriptive title as the top-level bookmark.
                # For "Page 176: Roadway: Drive Details", the full title is often better as the main bookmark.
                # Or you could just use `section_name + ": " + final_bookmark_title` if you want a prefix.
                writer.add_outline_item(
                    title=f"{section_name}: {final_bookmark_title}", # Combining for clear top-level
                    page_number=page_num - 1 # pypdf is 0-indexed
                )
            else:
                # If there are multiple items, create a parent bookmark (folder)
                first_item_page = section_items[0][0] # Page of the first item in this section
                parent_bookmark = writer.add_outline_item(title=section_name, page_number=first_item_page - 1)

                for page_num, title_full in section_items:
                    # Extract the specific title for the sub-bookmark
                    # This regex captures everything AFTER the second colon
                    bookmark_title_match = re.search(r':\s*[^:]+:\s*(.+)', title_full)
                    if bookmark_title_match:
                        sub_bookmark_title = bookmark_title_match.group(1).strip()
                    else:
                        # Fallback if the title doesn't have two colons,
                        # try to get the part after the first colon or use full title
                        first_colon_pos = title_full.find(':')
                        if first_colon_pos != -1:
                            sub_bookmark_title = title_full[first_colon_pos + 1:].strip()
                        else:
                            sub_bookmark_title = title_full

                    writer.add_outline_item(
                        title=sub_bookmark_title,
                        page_number=page_num - 1, # pypdf is 0-indexed
                        parent=parent_bookmark
                    )

        # Write the new PDF with all pages and added bookmarks to the output file
        with open(output_pdf_path, "wb") as output_file:
            writer.write(output_file)

        print(f"Bookmarks successfully added to: '{output_pdf_path}'")
        print(f"Original PDF: '{input_pdf_path}'")

    except FileNotFoundError:
        print(f"Error: Input PDF not found at '{input_pdf_path}'. Please check the path and ensure it's correct.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        print("Please ensure 'pypdf' is installed (`pip install pypdf`).")

In [93]:
if not os.path.exists(input_pdf_path):
    print(f"Error: The specified input PDF file does not exist: '{input_pdf_path}'")
else:
    add_bookmarks_to_pdf(input_pdf_path, output_pdf_path, toc)

In [102]:
structures_sheets = {key: value for key, value in labels.items() if value.split(':')[0] == 'Structures'}

In [103]:
structures_sheets

# Trying to Open to specific page

In [ ]:
import subprocess
import os
import win32api, win32con
from pathlib import Path

In [ ]:
def open_pdf_to_page_in_bluebeam(pdf_path, page_number):
    bluebeam_path = r"C:\Program Files\Bluebeam Software\Bluebeam Revu\21\Revu\Revu.exe"
    command = f'"{bluebeam_path}" "{pdf_path}" /A page={page_number}=OpenActions'

    try:
        subprocess.Popen(command, shell=True)
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
if __name__ == "__main__":
    pdf_file = r"X:\Bridge Design Resources\Projects\District 1\WYA\WYA-23-3.47\DRP - Comments\Backups\WYA-23-3.47 code testing.pdf"
    page_to_open = 35

    open_pdf_to_page_in_bluebeam(pdf_file, page_to_open)

In [ ]:
# ODOT's Posted Demonstration Plans
demo_plans = r"C:\Users\dparks1\PycharmProjects\civilpy\res\ODOT_sample_plans.pdf"

In [ ]:
pdf = pdfplumber.open(demo_plans)

In [ ]:
if "ELTIT" in title_page_text.split('\n') and "TEEHS" in title_page_text.split('\n'):
    print('Found')

In [ ]:
page = pdf.pages[13]
text = page.extract_text().split('\n')

In [ ]:
text

In [ ]:
sections = {}
current_section = None

for page_number, page in enumerate(pdf.pages, start=1):
    text = page.extract_text()
    if not text:
        continue  # Skip empty pages
    
    lines = text.split("\n")
    for line in lines:
        for keyword in section_keywords:
            if keyword in line:
                current_section = keyword
                if current_section not in sections:
                    sections[current_section] = []
                break
        
        if current_section:
            sections[current_section].append(line)